<a href="https://colab.research.google.com/github/lukassso/python-mini-projects/blob/master/Transform_transaction.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import logging

# Configure logging
logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(levelname)s - %(message)s')
logger = logging.getLogger()
logger.setLevel(logging.INFO)
logger.info("Logging initialized.")

INFO:root:Logging initialized.


**Authenticate the Colab Environment**

This step authenticates the Google Colab user to the connected Google Cloud project.
Authentication is required before performing any API calls involving BigQuery or Cloud Storage.

In [2]:
from google.colab import auth
auth.authenticate_user()

MessageError: Error: credential propagation was unsuccessful

### Create HMAC Key for Service Account

This block creates an HMAC access key and secret for the selected service account.

The HMAC key is required for DuckDB to authenticate with GCS when reading Parquet files over HTTPFS.


In [ ]:
from google.cloud import storage

# The project ID is usually part of your service account email (e.g., 'pydata-global' from 'pydata@pydata-global.iam.gserviceaccount.com')
storage_client = storage.Client(project='pydata-480521')

# Example: 'my-service-account@my-project.iam.gserviceaccount.com'
service_account_email = 'pydata@pydata-480521.iam.gserviceaccount.com'
print(f"Attempting to create HMAC key for: {service_account_email}")
try:
    # Create the HMAC key and secret
    hmac_key, secret = storage_client.create_hmac_key(service_account_email=service_account_email)

    key_id = hmac_key.access_id
    hmac_secret = secret

    logger.info(f"HMAC Key ID: {key_id}")
    logger.info(f"HMAC Secret: {hmac_secret}")
    logger.info("HMAC key created successfully. Store these credentials securely.")
except Exception as e:
    print(f"Error creating HMAC key: {e}")
    print("Please ensure:")
    error_msg = """
                  Please ensure:
                        1. The 'service_account_email' is correct.
                        2. The 'project' specified in storage.Client() is correct.
                        3. The authenticated user has 'Service Account Key Admin' role
                        (or similar) for the specified service account.
                        4. The service account itself has permissions to access the desired GCS bucket.
                    """
    logger.error({'error': str(e),
                    'message': str(error_msg)})

### Install Dependencies and Initialize DuckDB with HTTPFS

This cell installs DuckDB, gcsfs, and enables the HTTPFS extension needed for remote file access.


In [ ]:
!pip install duckdb gcsfs --quiet
import gcsfs
import duckdb

con = duckdb.connect()
con.execute("INSTALL httpfs;")
con.execute("LOAD httpfs;")

###  List Available Parquet Files in GCS
This step uses gcsfs to list the exported Solana Parquet files from storage bucket.
It retrieves the file paths and converts them into valid gs:// URIs for DuckDB.


In [ ]:
# list files
fs = gcsfs.GCSFileSystem()
paths = fs.glob("gs://raw-solana/exports/transactions/-*.parquet")

# convert to gs:// URIs
files = [f"gs://{p}" for p in paths]

# verify
logger.info(files[:3])  # should show strings




### Configure DuckDB with HMAC Credentials

Now, use the generated `key_id` and `hmac_secret` to configure DuckDB to access GCS. We will create a secret in DuckDB and then attempt to read the parquet files.

In [ ]:
# Create the HMAC key and secret
hmac_key, secret = storage_client.create_hmac_key(service_account_email=service_account_email)

key_id = hmac_key.access_id
hmac_secret = secret

logger.info({"HMAC Key ID: " : key_id})
logger.info({"HMAC Secret:" : hmac_secret})
con.execute(f"""
INSTALL httpfs; LOAD httpfs;

SET enable_http_logging = 1;

-- Provide GCS HMAC creds to DuckDB via a SECRET
CREATE OR REPLACE SECRET emr_gcs (
  TYPE gcs,
  KEY_ID '{key_id}',
  SECRET '{hmac_secret}'
);

""");

In [ ]:
files[0]

### Load Parquet Data into DuckDB DataFrame

This step reads a subset of the exported Solana transactions into a DuckDB DataFrame for exploration.
It verifies that remote reads and authentication are functioning correctly.

In [ ]:
df = con.execute(f"""
                SELECT * FROM read_parquet({files}) LIMIT 100
            """).df()

df.head(5)

### Create a DuckDB Table for Raw Transactions

Persist loaded Parquet data into a local DuckDB table named RAW_TRANSACTIONS.

In [ ]:
df_table = con.execute(f"""
                CREATE TABLE RAW_TRANSACTIONS AS
                SELECT * FROM read_parquet({files}) LIMIT 100
            """).df()

**DESCRIBE Table**

In [ ]:
con.sql("DESCRIBE RAW_TRANSACTIONS")


### Transform and Normalize Transaction Fields


This step creates a normalized table (transactions) from the raw Solana data.


In [ ]:
con.execute("""
CREATE TABLE transactions AS
SELECT
    signature AS transaction_signature,
    block_slot AS slot,
    block_timestamp AS block_time,
    recent_block_hash,
    fee,
    compute_units_consumed,
    accounts,
    log_messages,
    balance_changes,
    pre_token_balances,
    post_token_balances,
    err
FROM RAW_TRANSACTIONS;
""")


**Compute Fees per Hour**

In [ ]:
fee_per_hr = """
SELECT
  DATE_TRUNC('hour', block_time) AS hour ,
  SUM(fee) / 1e9 AS total_fees_sol
FROM transactions
GROUP BY hour
ORDER BY hour DESC;
"""
con.execute(fee_per_hr).df()



### Identify Active Signing Wallets

In [ ]:
active_signer = """
SELECT
  acc.unnest.pubkey AS wallet,
  COUNT(*) AS tx_count
FROM transactions,
UNNEST(accounts) AS acc
GROUP BY wallet
ORDER BY tx_count DESC
LIMIT 20;

"""

con.execute(active_signer).df()


In [ ]:
import pandas as pd
from google.cloud import bigquery

BQ_PROJECT = "YOUR_PROJECT_ID"
BQ_DATASET = "RAW_SOLANA"
BQ_TABLE_RAW = "transactions"
bq = bigquery.Client(project=BQ_PROJECT)


Write to Bigquery

In [ ]:
job = bq.query(fee_per_hr)
job.result()   # Wait for completion
print("Table created successfully.")


 Verify table exists

In [ ]:
table = bq.get_table("your_project.your_dataset.daily_fees")
print("Rows:", table.num_rows)
print("Schema:", table.schema)
